# Freight forecasting, step by step

This notebook is written for someone new to Python and machine learning. Run each cell from top to bottom. Each code cell does one small job, and the text explains what to look for.

We will load three recommended datasets:

1. **BDI**: a broad global dry-bulk freight signal.
2. **Brent oil**: an oil-price signal that can act as a rough fuel-cost proxy.
3. **India rainfall**: context for port-risk analysis.

The available BDI file is old and global. It is only useful for learning the workflow; it is not an East Coast India freight-rate dataset. Rainfall starts near the end of the BDI history, so we will inspect it but not combine it with this freight model.

## Step 1: Import the tools we need

These libraries help us read tables, create features, train a model, and save results.

In [1]:
from datetime import UTC, datetime
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

print('Imports are ready.')

Imports are ready.


## Step 2: Find the project files

This works whether Jupyter starts inside `ml/` or from the project folder above it.

In [2]:
from_here = Path.cwd()
if (from_here / 'data/raw').exists():
    ML_ROOT = from_here
else:
    ML_ROOT = from_here / 'ml'

DATA_DIR = ML_ROOT / 'data' / 'raw'
BDI_PATH = DATA_DIR / 'bdi_historical_1985_2009.csv'
BRENT_PATH = DATA_DIR / 'brent-daily.csv'
RAINFALL_PATH = DATA_DIR / 'daily-rainfall-at-state-level.csv'
MODEL_PATH = ML_ROOT / 'models' / 'bdi_proxy_model.joblib'
RESULTS_PATH = ML_ROOT / 'results.json'

print('ML folder:', ML_ROOT.resolve())
print('Raw data folder:', DATA_DIR.resolve())

ML folder: /Users/ankushkumarguptapahleja/Downloads/defacto_hackathon/ml
Raw data folder: /Users/ankushkumarguptapahleja/Downloads/defacto_hackathon/ml/data/raw


## Step 3: Check that all three files are present

If a file is missing, stop here and download it before moving on.

In [3]:
dataset_paths = [BDI_PATH, BRENT_PATH, RAINFALL_PATH]
for path in dataset_paths:
    print(path.name, '->', 'found' if path.exists() else 'MISSING')

if not all(path.exists() for path in dataset_paths):
    raise FileNotFoundError('One or more data files are missing from data/raw/.')

bdi_historical_1985_2009.csv -> found
brent-daily.csv -> found
daily-rainfall-at-state-level.csv -> found


## Step 4: Load the BDI file

`read_csv` loads a CSV file into a pandas table, called a DataFrame.

In [4]:
bdi_raw = pd.read_csv(BDI_PATH)
print('Rows and columns:', bdi_raw.shape)
display(bdi_raw.head())

Rows and columns: (6311, 2)


,x,y
0,1985-01-04,1000
1,1985-01-07,998
2,1985-01-08,996
3,1985-01-09,994
4,1985-01-10,980


The source uses `x` for the date and `y` for the BDI value. We rename them to names that are easier to understand.

In [5]:
bdi = pd.DataFrame({
    'date': pd.to_datetime(bdi_raw['x'], errors='coerce'),
    'bdi': pd.to_numeric(bdi_raw['y'], errors='coerce'),
})

display(bdi.head())

,date,bdi
0,1985-01-04,1000
1,1985-01-07,998
2,1985-01-08,996
3,1985-01-09,994
4,1985-01-10,980


Now remove rows that could not be read, sort dates from oldest to newest, and remove duplicate dates.

In [6]:
bdi = (
    bdi.dropna()
       .sort_values('date')
       .drop_duplicates('date')
       .reset_index(drop=True)
)

print('Clean BDI rows:', len(bdi))
print('BDI dates:', bdi['date'].min().date(), 'to', bdi['date'].max().date())

Clean BDI rows: 6311
BDI dates: 1985-01-04 to 2009-03-13


## Step 5: Load and clean the Brent oil file

Brent is a proxy for energy costs. It is not the price of marine bunker fuel.

In [7]:
brent_raw = pd.read_csv(BRENT_PATH)
print('Rows and columns:', brent_raw.shape)
display(brent_raw.head())

Rows and columns: (9092, 2)


,Date,Price
0,1987-05-20,18.63
1,1987-05-21,18.45
2,1987-05-22,18.55
3,1987-05-25,18.60
4,1987-05-26,18.63


In [8]:
brent = pd.DataFrame({
    'date': pd.to_datetime(brent_raw['Date'], errors='coerce'),
    'brent_usd_barrel': pd.to_numeric(brent_raw['Price'], errors='coerce'),
})

brent = (
    brent.dropna()
         .sort_values('date')
         .drop_duplicates('date')
         .reset_index(drop=True)
)

print('Clean Brent rows:', len(brent))
print('Brent dates:', brent['date'].min().date(), 'to', brent['date'].max().date())
display(brent.head())

Clean Brent rows: 9092
Brent dates: 1987-05-20 to 2026-09-22


,date,brent_usd_barrel
0,1987-05-20,18.63
1,1987-05-21,18.45
2,1987-05-22,18.55
3,1987-05-25,18.60
4,1987-05-26,18.63


## Step 6: Load and inspect India rainfall

Rainfall can help describe weather risk around ports. This source is aggregated by state, so it is a broad context feature.

In [9]:
rainfall_raw = pd.read_csv(RAINFALL_PATH)
print('Rows and columns:', rainfall_raw.shape)
print('Available columns:', list(rainfall_raw.columns))
display(rainfall_raw.head())

Rows and columns: (202234, 8)
Available columns: ['id', 'date', 'state_code', 'state_name', 'actual', 'rfs', 'normal', 'deviation']


,id,date,state_code,state_name,actual,rfs,normal,deviation
0,0,2009-01-01,5,Uttarakhand,0.0,0.003906,2.19,-100.0
1,1,2009-01-01,18,Assam,0.0,0.000000,0.52,-100.0
2,2,2009-01-01,16,Tripura,0.0,0.000000,0.09,-100.0
3,3,2009-01-01,36,Telangana,0.0,0.000000,0.17,-100.0
4,4,2009-01-01,2,Himachal Pradesh,0.0,0.008566,3.31,-100.0


In [10]:
rainfall = rainfall_raw.copy()
rainfall['date'] = pd.to_datetime(rainfall['date'], errors='coerce')
rainfall['actual'] = pd.to_numeric(rainfall['actual'], errors='coerce')
rainfall = rainfall.dropna(subset=['date', 'state_name', 'actual'])

print('Rainfall dates:', rainfall['date'].min().date(), 'to', rainfall['date'].max().date())
print('States in file:', rainfall['state_name'].nunique())

Rainfall dates: 2009-01-01 to 2023-12-26
States in file: 35


The ports in our problem are in Odisha, West Bengal, Andhra Pradesh, and Tamil Nadu. Keep those states for later port-risk analysis.

In [11]:
port_states = ['Odisha', 'West Bengal', 'Andhra Pradesh', 'Tamil Nadu']
rainfall_ports = rainfall[rainfall['state_name'].isin(port_states)].copy()

print('Rows for relevant port states:', len(rainfall_ports))
print('Matched state names:', sorted(rainfall_ports['state_name'].unique()))
display(rainfall_ports.head())

Rows for relevant port states: 21624
Matched state names: ['Andhra Pradesh', 'Odisha', 'Tamil Nadu', 'West Bengal']


,id,date,state_code,state_name,actual,rfs,normal,deviation
12,12,2009-01-01,28,Andhra Pradesh,0.00,0.000000,0.45,-100.0
23,23,2009-01-01,19,West Bengal,0.04,0.103831,0.02,100.0
31,31,2009-01-01,21,Odisha,0.00,0.000000,0.17,-100.0
32,32,2009-01-01,33,Tamil Nadu,0.00,0.000000,1.21,-100.0
49,49,2009-01-02,28,Andhra Pradesh,0.00,0.000000,0.53,-100.0


## Step 7: Record what each dataset contains

This table helps us remember the source, license note, and how we intend to use each file.

In [12]:
source_notes = pd.DataFrame([
    {'dataset': 'BDI', 'coverage': f"{bdi.date.min().date()} to {bdi.date.max().date()}", 'use': 'Global dry-bulk proxy', 'note': 'Community mirror; redistribution terms unverified'},
    {'dataset': 'Brent', 'coverage': f"{brent.date.min().date()} to {brent.date.max().date()}", 'use': 'Fuel-cost proxy', 'note': 'EIA crude oil; not bunker fuel'},
    {'dataset': 'India rainfall', 'coverage': f"{rainfall.date.min().date()} to {rainfall.date.max().date()}", 'use': 'Port-risk context', 'note': 'State-level; Open Data Commons Attribution'},
])
display(source_notes)

,dataset,coverage,use,note
0,BDI,1985-01-04 to 2009-03-13,Global dry-bulk proxy,Community mirror; redistribution terms unverified
1,Brent,1987-05-20 to 2026-09-22,Fuel-cost proxy,EIA crude oil; not bunker fuel
2,India rainfall,2009-01-01 to 2023-12-26,Port-risk context,State-level; Open Data Commons Attribution


## Step 8: Match BDI dates with Brent dates

The BDI and Brent tables do not have exactly the same dates. `merge_asof` matches each BDI row to the latest Brent price on or before that date. This avoids accidentally using a future oil price.

In [13]:
market = pd.merge_asof(
    bdi.sort_values('date'),
    brent.sort_values('date'),
    on='date',
    direction='backward',
).dropna(subset=['brent_usd_barrel']).reset_index(drop=True)

print('Rows with both BDI and Brent:', len(market))
print('Shared dates:', market['date'].min().date(), 'to', market['date'].max().date())
display(market.head())

Rows with both BDI and Brent: 5693
Shared dates: 1987-05-20 to 2009-03-13


,date,bdi,brent_usd_barrel
0,1987-05-20,1088,18.63
1,1987-05-21,1084,18.45
2,1987-05-22,1084,18.55
3,1987-05-25,1084,18.60
4,1987-05-26,1079,18.63


Rainfall begins in 2009, while this BDI file ends in March 2009. That gives us too little shared history to train a reliable freight model with rainfall, so we keep rainfall separate for a future port-risk model.

In [14]:
overlap_start = max(bdi['date'].min(), rainfall_ports['date'].min())
overlap_end = min(bdi['date'].max(), rainfall_ports['date'].max())
if overlap_start <= overlap_end:
    overlap_days = (overlap_end - overlap_start).days + 1
else:
    overlap_days = 0
print('Calendar days of BDI/rainfall overlap:', overlap_days)

Calendar days of BDI/rainfall overlap: 72


## Step 9: Create BDI history features

A lag is an older value. For example, `bdi_lag_5` is the BDI five observations before the current date.

In [15]:
features = pd.DataFrame(index=market.index)
features['bdi'] = market['bdi'].astype(float)
features['brent_usd_barrel'] = market['brent_usd_barrel'].astype(float)

for lag in (1, 5, 20, 60, 252):
    features[f'bdi_lag_{lag}'] = market['bdi'].shift(lag)

display(features.head(10))

,bdi,brent_usd_barrel,bdi_lag_1,bdi_lag_5,bdi_lag_20,bdi_lag_60,bdi_lag_252
0,1088.0,18.63,NaN,NaN,NaN,NaN,NaN
1,1084.0,18.45,1088.0,NaN,NaN,NaN,NaN
2,1084.0,18.55,1084.0,NaN,NaN,NaN,NaN
3,1084.0,18.60,1084.0,NaN,NaN,NaN,NaN
4,1079.0,18.63,1084.0,NaN,NaN,NaN,NaN
5,1075.0,18.60,1079.0,1088.0,NaN,NaN,NaN
6,1076.0,18.60,1075.0,1084.0,NaN,NaN,NaN
7,1070.0,18.58,1076.0,1084.0,NaN,NaN,NaN
8,1070.0,18.65,1070.0,1084.0,NaN,NaN,NaN
9,1047.0,18.68,1070.0,1079.0,NaN,NaN,NaN


A rolling mean averages recent values. A rolling standard deviation tells us how much recent values have moved around.

In [16]:
for window in (5, 20, 60, 252):
    features[f'bdi_mean_{window}'] = market['bdi'].rolling(window).mean()

for window in (20, 60):
    features[f'bdi_std_{window}'] = market['bdi'].rolling(window).std()

display(features[['bdi', 'bdi_mean_5', 'bdi_mean_20', 'bdi_std_20']].tail())

,bdi,bdi_mean_5,bdi_mean_20,bdi_std_20
5688,2262.0,2154.4,2029.25,105.176481
5689,2298.0,2207.2,2045.45,120.110684
5690,2271.0,2244.6,2056.25,130.293949
5691,2201.0,2251.4,2066.85,133.127669
5692,2122.0,2230.8,2077.55,128.197001


Percentage change features describe recent direction and size of movement.

In [17]:
for window in (5, 20, 60):
    features[f'bdi_change_{window}'] = market['bdi'].pct_change(window)

for lag in (1, 5, 20):
    features[f'brent_lag_{lag}'] = market['brent_usd_barrel'].shift(lag)

for window in (5, 20, 60):
    features[f'brent_change_{window}'] = market['brent_usd_barrel'].pct_change(window)

display(features.tail())

,bdi,brent_usd_barrel,bdi_lag_1,bdi_lag_5,bdi_lag_20,bdi_lag_60,bdi_lag_252,bdi_mean_5,bdi_mean_20,bdi_mean_60,...,bdi_std_60,bdi_change_5,bdi_change_20,bdi_change_60,brent_lag_1,brent_lag_5,brent_lag_20,brent_change_5,brent_change_20,brent_change_60
5688,2262.0,44.55,2225.0,2014.0,1815.0,803.0,7684.0,2154.4,2029.25,1306.866667,...,554.119655,0.123138,0.246281,1.816936,43.48,42.60,47.23,0.045775,-0.056744,-0.010440
5689,2298.0,44.99,2262.0,2034.0,1974.0,828.0,7684.0,2207.2,2045.45,1331.366667,...,564.979135,0.129794,0.164134,1.775362,44.55,42.72,45.88,0.053137,-0.019398,0.071190
5690,2271.0,43.20,2298.0,2084.0,2055.0,836.0,7684.0,2244.6,2056.25,1355.283333,...,573.955738,0.089731,0.105109,1.716507,44.99,46.07,44.24,-0.062297,-0.023508,0.032505
5691,2201.0,42.19,2271.0,2167.0,1989.0,829.0,7619.0,2251.4,2066.85,1378.150000,...,579.932428,0.015690,0.106586,1.655006,43.20,44.45,47.23,-0.050844,-0.106712,0.049764
5692,2122.0,44.97,2201.0,2225.0,1908.0,818.0,7679.0,2230.8,2077.55,1399.883333,...,583.010599,-0.046292,0.112159,1.594132,42.19,43.48,43.36,0.034269,0.037131,0.137905


Month-of-year patterns can matter in shipping. These sine and cosine columns represent the time of year in a form models can use.

In [18]:
day_of_year = market['date'].dt.dayofyear
features['year_sin'] = np.sin(2 * np.pi * day_of_year / 365.25)
features['year_cos'] = np.cos(2 * np.pi * day_of_year / 365.25)

print('Feature columns:', len(features.columns))
display(features.head())

Feature columns: 24


,bdi,brent_usd_barrel,bdi_lag_1,bdi_lag_5,bdi_lag_20,bdi_lag_60,bdi_lag_252,bdi_mean_5,bdi_mean_20,bdi_mean_60,...,bdi_change_20,bdi_change_60,brent_lag_1,brent_lag_5,brent_lag_20,brent_change_5,brent_change_20,brent_change_60,year_sin,year_cos
0,1088.0,18.63,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.669290,-0.743001
1,1084.0,18.45,1088.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,18.63,NaN,NaN,NaN,NaN,NaN,0.656411,-0.754404
2,1084.0,18.55,1084.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,18.45,NaN,NaN,NaN,NaN,NaN,0.643337,-0.765584
3,1084.0,18.60,1084.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,18.55,NaN,NaN,NaN,NaN,NaN,0.602988,-0.797750
4,1079.0,18.63,1084.0,NaN,NaN,NaN,NaN,1083.8,NaN,NaN,...,NaN,NaN,18.60,NaN,NaN,NaN,NaN,NaN,0.589176,-0.808005


## Step 10: Create the answer the model should learn

The target is BDI five observations in the future. `shift(-5)` moves future values upward so they line up with today’s features.

In [19]:
HORIZON = 5
target = market['bdi'].shift(-HORIZON).rename('target_bdi')
display(pd.DataFrame({'date': market['date'], 'today_bdi': market['bdi'], 'bdi_in_5_observations': target}).head(10))

,date,today_bdi,bdi_in_5_observations
0,1987-05-20,1088,1075.0
1,1987-05-21,1084,1076.0
2,1987-05-22,1084,1070.0
3,1987-05-25,1084,1070.0
4,1987-05-26,1079,1047.0
5,1987-05-27,1075,1024.0
6,1987-05-28,1076,1015.0
7,1987-05-29,1070,990.0
8,1987-06-01,1070,986.0
9,1987-06-02,1047,986.0


The first rows have missing history because lag features need earlier observations. The last rows have no future target. Remove those rows before training.

In [20]:
features = features.replace([np.inf, -np.inf], np.nan)
samples = features.join(target).dropna()

print('Usable training examples:', len(samples))
display(samples.head())

Usable training examples: 5436


,bdi,brent_usd_barrel,bdi_lag_1,bdi_lag_5,bdi_lag_20,bdi_lag_60,bdi_lag_252,bdi_mean_5,bdi_mean_20,bdi_mean_60,...,bdi_change_60,brent_lag_1,brent_lag_5,brent_lag_20,brent_change_5,brent_change_20,brent_change_60,year_sin,year_cos,target_bdi
252,1376.0,16.45,1344.0,1358.0,1584.0,1547.0,1088.0,1352.4,1437.65,1540.200000,...,-0.110537,16.15,16.60,15.55,-0.009036,0.057878,0.044444,0.817401,-0.576069,1445.0
253,1387.0,16.50,1376.0,1358.0,1572.0,1544.0,1084.0,1358.2,1428.40,1537.583333,...,-0.101684,16.45,15.95,16.20,0.034483,0.018519,0.052296,0.786597,-0.617467,1446.0
254,1391.0,16.38,1387.0,1345.0,1560.0,1546.0,1084.0,1367.4,1419.95,1535.000000,...,-0.100259,16.50,16.08,16.48,0.018657,-0.006068,0.047985,0.775859,-0.630907,1452.0
255,1398.0,16.48,1391.0,1339.0,1544.0,1562.0,1084.0,1379.2,1412.65,1532.266667,...,-0.104994,16.38,16.15,16.55,0.020433,-0.004230,0.039748,0.764891,-0.644159,1456.0
256,1428.0,16.40,1398.0,1344.0,1542.0,1566.0,1079.0,1396.0,1406.95,1529.966667,...,-0.088123,16.48,16.15,16.65,0.015480,-0.015015,0.059432,0.753698,-0.657221,1438.0


## Step 11: Split by time

We train on the first 80% of dates and evaluate on the later 20%. For time-series forecasting, do not shuffle rows: future observations must stay out of training.

In [21]:
split_at = int(len(samples) * 0.8)
train = samples.iloc[:split_at]
test = samples.iloc[split_at:]

feature_columns = [column for column in samples.columns if column != 'target_bdi']
x_train = train[feature_columns]
y_train = train['target_bdi']
x_test = test[feature_columns]
y_test = test['target_bdi']

print('Training examples:', len(train))
print('Test examples:', len(test))
print('First test date:', market.loc[test.index[0], 'date'].date())

Training examples: 4348
Test examples: 1088
First test date: 2005-01-05


## Step 12: Create and train the model

Gradient boosting builds many small decision trees. Each tree tries to improve on the errors made by earlier trees.

In [22]:
model = HistGradientBoostingRegressor(
    max_iter=250,
    learning_rate=0.06,
    max_leaf_nodes=15,
    l2_regularization=1.0,
    random_state=42,
)
print('Model is ready.')

Model is ready.


In [23]:
model.fit(x_train, y_train)
print('Training finished.')

Training finished.


## Step 13: Make predictions

The model predicts the BDI five observations after each date in the test period.

In [24]:
predictions = model.predict(x_test)
print('Predictions made:', len(predictions))
print('First five predictions:', predictions[:5])

Predictions made: 1088
First five predictions: [4607.96345401 4608.26348556 4653.06112486 4724.34719711 4989.82153204]


A simple baseline is important. Persistence means: predict that the BDI five observations from now will stay at today’s value.

In [25]:
baseline_predictions = test['bdi'].to_numpy()
print('First five baseline predictions:', baseline_predictions[:5])

First five baseline predictions: [4395. 4405. 4485. 4627. 4726.]


## Step 14: Compare errors

MAE is the average size of the prediction error. Lower is better. We compare the model with persistence.

In [26]:
def calculate_metrics(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    return {
        'MAE': float(mean_absolute_error(actual, predicted)),
        'RMSE': float(mean_squared_error(actual, predicted) ** 0.5),
        'MAPE percent': float(np.mean(np.abs((actual - predicted) / actual)) * 100),
    }

model_metrics = calculate_metrics(y_test, predictions)
baseline_metrics = calculate_metrics(y_test, baseline_predictions)
comparison = pd.DataFrame([model_metrics, baseline_metrics], index=['Model', 'Persistence baseline'])
display(comparison)

,MAE,RMSE,MAPE percent
Model,1312.768719,2108.348841,22.329442
Persistence baseline,251.272059,381.342383,5.931389


If the model error is larger than the baseline error, the model has not learned a useful forecasting pattern from this dataset. Do not use it for a real chartering decision.

In [27]:
candidate_beats_baseline = model_metrics['MAE'] < baseline_metrics['MAE']
print('Does the model beat persistence?', candidate_beats_baseline)
print('Suitable for procurement decisions? No — this is an old global BDI proxy.')

Does the model beat persistence? False
Suitable for procurement decisions? No — this is an old global BDI proxy.


## Step 15: Save the model and a clear record of the run

The model file can be loaded later. The JSON file records data coverage, scores, and known limitations.

In [28]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump({
    'model': model,
    'feature_columns': feature_columns,
    'horizon_observations': HORIZON,
}, MODEL_PATH)
print('Saved model to:', MODEL_PATH)

Saved model to: /Users/ankushkumarguptapahleja/Downloads/defacto_hackathon/ml/models/bdi_proxy_model.joblib


In [29]:
report = {
    'status': 'prototype_only',
    'target': 'global Baltic Dry Index, five observations ahead',
    'data_sources': {
        'bdi': {
            'file': str(BDI_PATH.relative_to(ML_ROOT)),
            'start': bdi['date'].min().date().isoformat(),
            'end': bdi['date'].max().date().isoformat(),
            'source': 'Community GitHub mirror; redistribution terms unverified',
        },
        'brent': {
            'file': str(BRENT_PATH.relative_to(ML_ROOT)),
            'start': brent['date'].min().date().isoformat(),
            'end': brent['date'].max().date().isoformat(),
            'source': 'EIA Brent crude oil; proxy, not marine bunker fuel',
        },
        'rainfall': {
            'file': str(RAINFALL_PATH.relative_to(ML_ROOT)),
            'start': rainfall['date'].min().date().isoformat(),
            'end': rainfall['date'].max().date().isoformat(),
            'source': 'India Data Portal / India-WRIS; Open Data Commons Attribution',
            'role': 'loaded and inspected; not used to train due short overlap',
        },
    },
    'training_rows': int(len(train)),
    'test_rows': int(len(test)),
    'model_metrics': model_metrics,
    'persistence_baseline_metrics': baseline_metrics,
    'candidate_beats_baseline': bool(candidate_beats_baseline),
    'recommended_for_decision_use': False,
    'model_file': str(MODEL_PATH.relative_to(ML_ROOT)),
    'trained_at_utc': datetime.now(UTC).isoformat(),
    'limitations': [
        'The BDI is a global benchmark, not an East Coast India route quote.',
        'The available BDI file ends in 2009.',
        'Brent is crude oil, not marine bunker fuel.',
        'Rainfall is state-level and its overlap with this BDI file is too short for training.',
        'Use licensed India-route fixture/assessment data before decision use.',
    ],
}
RESULTS_PATH.write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
print('Saved results to:', RESULTS_PATH)
print(json.dumps(report, indent=2))

Saved results to: /Users/ankushkumarguptapahleja/Downloads/defacto_hackathon/ml/results.json
{
  "status": "prototype_only",
  "target": "global Baltic Dry Index, five observations ahead",
  "data_sources": {
    "bdi": {
      "file": "data/raw/bdi_historical_1985_2009.csv",
      "start": "1985-01-04",
      "end": "2009-03-13",
      "source": "Community GitHub mirror; redistribution terms unverified"
    },
    "brent": {
      "file": "data/raw/brent-daily.csv",
      "start": "1987-05-20",
      "end": "2026-09-22",
      "source": "EIA Brent crude oil; proxy, not marine bunker fuel"
    },
    "rainfall": {
      "file": "data/raw/daily-rainfall-at-state-level.csv",
      "start": "2009-01-01",
      "end": "2023-12-26",
      "source": "India Data Portal / India-WRIS; Open Data Commons Attribution",
      "role": "loaded and inspected; not used to train due short overlap"
    }
  },
  "training_rows": 4348,
  "test_rows": 1088,
  "model_metrics": {
    "MAE": 1312.768718719959,